### Performing EDA on OLIST Dataset

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as snb

In [40]:
customer_data = pd.read_csv("..\data\olist_customers_dataset.csv")
olist_orders = pd.read_csv("..\data\olist_orders_dataset.csv")
olist_order_items = pd.read_csv("..\data\olist_order_items_dataset.csv")
olist_order_payments = pd.read_csv("..\data\olist_order_payments_dataset.csv")
olist_order_reviews = pd.read_csv("..\data\olist_order_reviews_dataset.csv")
olist_products = pd.read_csv("..\data\olist_products_dataset.csv")
olist_product_category = pd.read_csv("..\data\product_category_name_translation.csv")

In [41]:
customer_data.head()
olist_orders.head()
olist_order_items.head()
# olist_order_payments.head()
# # olist_order_reviews.head()
# olist_products.head()
# olist_product_category['product_category_name'].nunique()


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


### Checking Multiple Payments issue

In [42]:

counts = olist_order_payments["order_id"].value_counts()
duplicate_ids = counts[counts>1].index
olist_order_payments[olist_order_payments["order_id"].isin(duplicate_ids)].sort_values(by="order_id")

#Faster Way
olist_order_payments[olist_order_payments.duplicated(subset=['order_id'], keep=False)].sort_values(by='order_id')

,order_id,payment_sequential,payment_type,payment_installments,payment_value
80856,0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
89575,0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63
20036,002f19a65a2ddd70a090297872e6d64e,1,voucher,1,44.11
98894,002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
30155,0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
...,...,...,...,...,...
21648,ffa1dd97810de91a03abd7bd76d2fed1,2,voucher,1,418.73
32912,ffa39020fe7c8a3e907320e1bec4b985,1,credit_card,1,7.13
3009,ffa39020fe7c8a3e907320e1bec4b985,2,voucher,1,64.01
75188,ffc730a0615d28ec19f9cad02cb41442,1,credit_card,1,14.76


In [43]:
# Grouping multiple orders in payments
unique_order_payments = olist_order_payments.groupby(["order_id"]).agg(max_installments=('payment_installments', 'max'),
                                                                       max_methods = ('payment_sequential', 'max'),
                                                                       total_payment = ('payment_value','sum')).reset_index()
primary_payments=olist_order_payments.sort_values(by="payment_value", ascending=False).drop_duplicates(subset="order_id")
primary_payments

# Final Cleaned Payments Data
olist_cleaned_payments = pd.merge(primary_payments[['order_id','payment_type']], unique_order_payments,
                                  on="order_id", how="inner")
olist_cleaned_payments.head(26)

,order_id,payment_type,max_installments,max_methods,total_payment
0,03caa2c082116e1d31e67e9ae3700499,credit_card,1,1,13664.08
1,736e1922ae60d0d6a89247b851902527,boleto,1,1,7274.88
2,0812eb902a67711a1cb742b3cdaa65ae,credit_card,8,1,6929.31
3,fefacc66af859508bf1a7934eab1e97f,boleto,1,1,6922.21
4,f5136e38d1a14a4dbd87dff67da82701,boleto,1,1,6726.66
5,2cc9089445046817a7539d90805e6e5a,boleto,1,1,6081.54
6,a96610ab360d42a2e5335a3998b4718a,credit_card,10,1,4950.34
7,b4c4b76c642808cbe472a32b86cddc95,credit_card,5,1,4809.44
8,199af31afc78c699f0dbf71fb178d4d4,credit_card,8,1,4764.34
9,8dbc85d1447242f3b127dda390d56e19,credit_card,8,1,4681.78


### Merging customer and order data

In [44]:

customer_order_data = pd.merge(customer_data, olist_orders, how="left", on="customer_id")
customer_order_data.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05 00:00:00
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06 00:00:00
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13 00:00:00
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10 00:00:00
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15 00:00:00


#### pd.merge ignores Index!!

In [45]:
# Merging Customer and Payment data
master_customer_data = pd.merge(customer_order_data, olist_cleaned_payments, on="order_id", how="inner")
master_customer_data.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_type,max_installments,max_methods,total_payment
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05 00:00:00,credit_card,2,1,146.87
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06 00:00:00,credit_card,8,1,335.48
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13 00:00:00,credit_card,7,1,157.73
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10 00:00:00,credit_card,1,1,173.30
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15 00:00:00,credit_card,8,1,252.25


### order_items and product data handling

.nunique(), .duplicated(), .isin() return series. To convert to dataframe -> .reset_index()

.isnull(), .isna() returns Dataframe

In [46]:
olist_order_items[olist_order_items.duplicated(subset=["order_id"],keep=False)].head(10)
olist_order_items[olist_order_items["order_item_id"]>20]
olist_order_items[olist_order_items["order_id"]=="ffb9a9cd00c74c11c24aa30b3d78e03b"]
# There are order_ids which have Different products!!!

mixed_product_counts = olist_order_items.groupby('order_id')['product_id'].nunique()  
mixed_product_order_ids = mixed_product_counts[mixed_product_counts>1].index
mixed_product_orders = olist_order_items[olist_order_items['order_id'].isin(mixed_product_order_ids)]
mixed_product_orders.head(25)
mixed_product_orders.sort_values(by=['order_id', 'order_item_id']).head(6)
# All different products in an order!!

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
80,002f98c0f7efd42638ed6100ca699b42,1,d41dc2f2979f52d75d78714b378d4068,7299e27ed73d2ad986de7f7c77d919fa,2017-08-10 09:30:15,8.99,32.57
81,002f98c0f7efd42638ed6100ca699b42,2,880be32f4db1d9f6e2bec38fb6ac23ab,fa40cc5b934574b62717c68f3d678b6d,2017-08-10 09:30:15,44.90,7.16
91,00337fe25a3780b3424d9ad7c5a4b35e,1,1f9799a175f50c9fa725984775cac5c5,cfb1a033743668a192316f3c6d1d2671,2017-09-29 17:50:16,59.90,9.94
92,00337fe25a3780b3424d9ad7c5a4b35e,2,13944d17b257432717fd260e69853140,cfb1a033743668a192316f3c6d1d2671,2017-09-29 17:50:16,59.90,9.94
151,005d9a5423d47281ac463a968b3936fb,1,fb7a100ec8c7b34f60cec22b1a9a10e0,d98eec89afa3380e14463da2aabaea72,2017-10-24 12:28:16,49.99,18.12
152,005d9a5423d47281ac463a968b3936fb,2,4c3ae5db49258df0784827bdacf3b396,d98eec89afa3380e14463da2aabaea72,2017-10-24 12:28:16,24.99,13.58


##### Merging related product tables: Order_items, products, product_category_name

Checking nulls -> 1.isna().sum(), 2. isna().any(), isna.any(axis=1) checks horizontally

Filling nulls -> .fillna(df_column) only corresponding row/index of df_column will be filled!!

In [47]:
olist_detailed_order = pd.merge(olist_order_items, olist_products, how="left", on="product_id")
olist_product_category.head()
olist_detailed_order = pd.merge(olist_detailed_order, olist_product_category, how="left", on="product_category_name")
olist_detailed_order = olist_detailed_order.drop(["product_name_lenght", "product_description_lenght",
                                                  "product_photos_qty","product_weight_g",
                                                  "product_length_cm","product_width_cm","product_height_cm"],axis=1)
olist_detailed_order[olist_detailed_order['order_id']=="ffb9a9cd00c74c11c24aa30b3d78e03b"]

#Checking null values
olist_detailed_order.isna().sum(axis=0)
# Filling null valuees

olist_detailed_order['product_category_name_english'] = olist_detailed_order['product_category_name_english'].fillna(
                                                                            olist_detailed_order['product_category_name']).fillna(
                                                                                'Unknown'
                                                                            )
                     
olist_detailed_order = olist_detailed_order.drop(["product_category_name"], axis=1)
olist_detailed_order = olist_detailed_order.rename(columns={"product_category_name_english":"product_category_name"})
olist_detailed_order.isnull().sum()
olist_detailed_order.head()
olist_detailed_order.shape


(112650, 8)

In [48]:
# Handling Multiple Orders with Mixed Products
mixed_category_products_counts = olist_detailed_order.groupby('order_id')['product_category_name'].nunique()
mixed_category_index = mixed_category_products_counts[mixed_category_products_counts>1].index
mixed_category_products = olist_detailed_order[olist_detailed_order['order_id'].isin(mixed_category_index)]
mixed_category_products

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name
80,002f98c0f7efd42638ed6100ca699b42,1,d41dc2f2979f52d75d78714b378d4068,7299e27ed73d2ad986de7f7c77d919fa,2017-08-10 09:30:15,8.99,32.57,consoles_games
81,002f98c0f7efd42638ed6100ca699b42,2,880be32f4db1d9f6e2bec38fb6ac23ab,fa40cc5b934574b62717c68f3d678b6d,2017-08-10 09:30:15,44.90,7.16,toys
151,005d9a5423d47281ac463a968b3936fb,1,fb7a100ec8c7b34f60cec22b1a9a10e0,d98eec89afa3380e14463da2aabaea72,2017-10-24 12:28:16,49.99,18.12,toys
152,005d9a5423d47281ac463a968b3936fb,2,4c3ae5db49258df0784827bdacf3b396,d98eec89afa3380e14463da2aabaea72,2017-10-24 12:28:16,24.99,13.58,baby
153,005d9a5423d47281ac463a968b3936fb,3,4c3ae5db49258df0784827bdacf3b396,d98eec89afa3380e14463da2aabaea72,2017-10-24 12:28:16,24.99,13.58,baby
...,...,...,...,...,...,...,...,...
112500,ffa5e4c604dea4f0a59d19cc2322ac19,1,59d8034bb2a82cd5092e78d42148fa44,59cd88080b93f3c18508673122d26169,2017-12-11 08:41:20,29.99,15.10,pet_shop
112501,ffa5e4c604dea4f0a59d19cc2322ac19,2,bd421826916d3e1d445cb860cea3c0fb,59cd88080b93f3c18508673122d26169,2017-12-11 08:41:20,29.99,15.10,Unknown
112530,ffb8f7de8940249a3221252818937ecb,1,bd6e8cf9fe4122c385da2bcb9f979d5d,9f50216bfd01913736a55a11b55ea842,2018-07-27 09:04:32,45.00,6.79,telephony
112531,ffb8f7de8940249a3221252818937ecb,2,803f77475e1b51b47f1bfec4f2ec353f,c9c7905cffc4ef9ff9f113554423e671,2018-07-25 09:04:32,79.99,6.37,auto


### Challenge: Multiple Category Products in same order_id.
Solution: Create One-Hot encoding of broader industry categories, i.e, 71 categories mapped -> 8 broad categories

In [49]:
# Biggest Hurdle: Multiple Category Products
broad_category_dict = {
    'health_beauty': 'Health_Beauty', 'perfumery': 'Health_Beauty',
    'computers_accessories': 'Technology', 'telephony': 'Technology', 'electronics': 'Technology', 'pc_gamer': 'Technology',
    'bed_bath_table': 'Home_Furniture', 'furniture_decor': 'Home_Furniture', 'housewares': 'Home_Furniture', 'office_furniture': 'Home_Furniture', 'garden_tools': 'Home_Furniture',
    'sports_leisure': 'Sports_Leisure',
    'toys': 'Kids_Baby', 'baby': 'Kids_Baby', 'diapers_and_hygiene': 'Kids_Baby',
    'auto': 'Auto',
    'watches_gifts': 'Fashion_Accessories', 'fashion_bags_accessories': 'Fashion_Accessories', 'luggage_accessories': 'Fashion_Accessories',
    'pet_shop': 'Pets',
    'books_technical': 'Academic', 'food': 'Health_Beauty', 'home_construction': 'Home_Furniture'
}

olist_detailed_order['broad_category_name'] = olist_detailed_order['product_category_name'].map(broad_category_dict).fillna('Other')
olist_detailed_order

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,broad_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,Other
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,Pets
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,furniture_decor,Home_Furniture
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumery,Health_Beauty
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,garden_tools,Home_Furniture
...,...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,housewares,Home_Furniture
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,computers_accessories,Technology
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,sports_leisure,Sports_Leisure
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,computers_accessories,Technology


#### pd.crosstab(): Creates a category matrix between 2 columns whose relationship is to be determined.
Here, relationship between each order_id and product category was desired, hence using crosstab. Hence grouping + Duplication removal occurs in one go

#### pd.get_dummies(): Creates a row for every found category.
Hence duplicate categories would remain and further .groupby().max() would have to be performed

In [50]:
category_matrix = pd.crosstab(olist_detailed_order['order_id'], olist_detailed_order['broad_category_name'])

olist_order_categories = category_matrix.clip(upper=1)
olist_order_categories=olist_order_categories.reset_index()
olist_order_categories.columns.name=None
olist_order_categories.head()

# Data Shape mismatch!! 98666 records!! against 99400

,order_id,Academic,Auto,Fashion_Accessories,Health_Beauty,Home_Furniture,Kids_Baby,Other,Pets,Sports_Leisure,Technology
0,00010242fe8c5a6d1ba2dd792cb16214,0,0,0,0,0,0,1,0,0,0
1,00018f77f2f0320c557190d7a144bdd3,0,0,0,0,0,0,0,1,0,0
2,000229ec398224ef6ca0657da4fc703e,0,0,0,0,1,0,0,0,0,0
3,00024acbcdf0a6daa1e931b038114c75,0,0,0,1,0,0,0,0,0,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,0,0,0,0,1,0,0,0,0,0


In [51]:
# Checking descrepancy
master_customer_data[master_customer_data['order_status']!="delivered"].value_counts(subset='order_status')
# Cancelled and some more orders are not in olist_order_categories!! Hence 774 less orders


order_status
shipped        1107
canceled        625
unavailable     609
invoiced        314
processing      301
created           5
approved          2
Name: count, dtype: int64

#### Final Merging: Main(Master_customer), secondary(cleaned_payments, product_categories)

In [70]:
final_olist_order_dataset = pd.merge(master_customer_data, olist_order_categories, how="left", on="order_id")
final_olist_order_dataset[final_olist_order_dataset['max_methods']>1]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,Academic,Auto,Fashion_Accessories,Health_Beauty,Home_Furniture,Kids_Baby,Other,Pets,Sports_Leisure,Technology
18,9b8ce803689b3562defaad4613ef426f,7f3a72e8f988c6e735ba118d54f47458,5416,sao paulo,SP,17825f24877a9289214c301ae0c9424b,delivered,2017-05-11 13:48:47,2017-05-13 11:55:16,2017-05-15 15:30:02,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
34,469634941c27cd844170935a3cf60b95,ef07ba9aa5226f77264ffa5762b2280b,81750,curitiba,PR,a9119eb77d6200811953803a7b6539e1,delivered,2018-03-12 13:07:03,2018-03-12 13:15:28,2018-03-13 22:12:18,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
54,8247b5583327ab8be19f96e1fb82f77b,d85547cd859833520b311b4458a14c1c,23970,parati,RJ,a6917b5d71e0e9bc434e9228db8daeb2,delivered,2017-06-09 15:46:17,2017-06-10 15:42:38,2017-06-12 17:10:55,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
65,a02f66c3af7b16eec19ddcd98b645fe3,b3548d0cec408ae13d143bb4eeebaa6c,13323,salto,SP,db97652cf517d2cd03db63dec489ca62,delivered,2017-10-01 08:57:03,2017-10-01 09:14:07,2017-10-02 19:32:57,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
83,91ec76836092bba85d11761078ed7bb5,6edd17d0a29e2d4057e694afee5eaa3b,28010,campos dos goytacazes,RJ,9cefab6270eb935eb96a97c56b8e7984,delivered,2018-06-07 21:41:19,2018-06-07 21:54:43,2018-06-11 12:16:00,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99372,39794761aa663d624636cdc242f1950f,057583d44728a47f6f223414b7e6cc83,11730,mongagua,SP,c15b6194f063aa18a0fd47136445c6ec,delivered,2018-05-11 14:27:42,2018-05-11 14:58:05,2018-05-14 14:10:00,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
99417,30c96385d694acb8aa2dc0df1770120b,b96d6a178adbabf269fd843b37327798,26112,belford roxo,RJ,357b4b724bbf34f1d64b1c5dfdc88120,delivered,2018-01-24 02:22:12,2018-01-24 15:57:52,2018-01-26 21:28:37,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
99425,935993f47af1ed7d0715c26b686341c5,4452b8ef472646c4cc042cb31a291f3b,12236,sao jose dos campos,SP,e86b1b2dd48839d7351406434afb578d,delivered,2017-11-25 00:31:20,2017-11-25 02:33:11,2017-11-27 18:24:54,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
99428,1ed0c832c2dd99570a59260e71768bdf,82d46759af0369aad49084bacf85a6c3,37610,bom repouso,MG,51c6d2f460589fa7b65f2da51e860206,delivered,2017-11-14 12:04:09,2017-11-14 12:15:25,2017-11-27 20:44:47,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


##### Till now Unique Order data has been created, not unique Customer.
Hence For customer segmentation, grouping by customer is important!!

In [71]:

final_olist_order_dataset[final_olist_order_dataset.duplicated(subset="customer_unique_id", keep=False)].sort_values(by="customer_unique_id").head()



,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,Academic,Auto,Fashion_Accessories,Health_Beauty,Home_Furniture,Kids_Baby,Other,Pets,Sports_Leisure,Technology
35607,24b0e2bd287e47d54d193e7bbb51103f,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA,c306eca42d32507b970739b5b6a5a33a,canceled,2018-08-13 09:14:07,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19299,1afe8a9c67eec3516c09a8bdcc539090,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA,bb874c45df1a3c97842d52f31efee99a,delivered,2018-07-28 00:23:49,2018-07-28 00:35:19,2018-07-31 15:57:00,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
20023,1b4a75b3478138e99902678254b260f4,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ,a61d617fbe5bd006e40d3a0988fc844b,delivered,2017-07-27 14:13:03,2017-07-27 14:25:14,2017-07-28 17:45:36,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
22065,f6efe5d5c7b85e12355f9d5c3db46da2,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ,08204559bebd39e09ee52dcb56d8faa2,delivered,2018-01-14 07:36:54,2018-01-14 07:49:28,2018-01-16 16:39:34,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
72450,49cf243e0d353cd418ca77868e24a670,004b45ec5c64187465168251cd1c9c2f,57055,maceio,AL,90ae229a4addcfead792e2564554f09c,shipped,2017-09-01 12:11:23,2017-09-05 04:30:20,2017-09-08 19:42:16,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


##### Creating Recency Metric:
Most_Recent_Order_date - Order_purchase_date

In [72]:
final_olist_order_dataset['order_purchase_timestamp'] = pd.to_datetime(final_olist_order_dataset['order_purchase_timestamp'])
most_recent_date = final_olist_order_dataset['order_purchase_timestamp'].max()

final_olist_order_dataset['days_last_purchased'] = most_recent_date - final_olist_order_dataset['order_purchase_timestamp']
final_olist_order_dataset['days_last_purchased'] = final_olist_order_dataset['days_last_purchased'].dt.days
final_olist_order_dataset.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,Auto,Fashion_Accessories,Health_Beauty,Home_Furniture,Kids_Baby,Other,Pets,Sports_Leisure,Technology,days_last_purchased
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,519
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,277
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,151
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,218
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,80


#### Creating Frequency Metric: max(order_item_id)

In [73]:
# Finding Maximum orders per order_id
order_counts_per_order = olist_detailed_order.groupby(['order_id']).agg(num_orders=("order_item_id","max")).reset_index()
order_counts_per_order[order_counts_per_order['order_id']=="00526a9d4ebde463baee25f386963ddc"]
order_counts_per_order.shape

# Joining with final_order_data
final_olist_order_dataset = pd.merge(final_olist_order_dataset, order_counts_per_order,
                                     how="left", on="order_id")
final_olist_order_dataset.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,Fashion_Accessories,Health_Beauty,Home_Furniture,Kids_Baby,Other,Pets,Sports_Leisure,Technology,days_last_purchased,num_orders
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,519,1.0
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,277,1.0
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,151,1.0
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,218,1.0
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,80,1.0


#### Final Cleaning and Grouping of Dataset

In [74]:
# Keeping only delivered columns since contributing only to revenue

#NOTE:1. order_status shipped, invoice,etc. also contributed to revenue but have been too long(more than 10 days)
      #2. This being a historical dataset, eventually means such orders were finally cancelled!!

final_olist_order_dataset = final_olist_order_dataset[final_olist_order_dataset['order_status']=="delivered"]
#Shape: 96477
final_olist_order_dataset = final_olist_order_dataset.drop(labels=["order_status", "order_purchase_timestamp",
                                                                    "order_approved_at","order_delivered_carrier_date",
                                                                    'order_delivered_carrier_date','order_delivered_customer_date',
                                                                    'order_estimated_delivery_date'], axis=1)

#### Grouping rules for customer
Number_of_unique orders = Frequency
Each Category whether purchased or not => max()[acting as "or" operator]

In [76]:
# Taking care of needed columns after .groupby() ddestroys them
category_cols = [
    'Academic', 'Auto', 'Fashion_Accessories', 'Health_Beauty', 
    'Home_Furniture', 'Kids_Baby', 'Other', 'Pets', 
    'Sports_Leisure', 'Technology'
]

agg_rules = {
    "days_last_purchased": 'min',
    "order_id": 'nunique',
    "total_payment":'sum'
}
for cols in category_cols:
    agg_rules[cols] = 'max'
agg_rules    

{'days_last_purchased': 'min',
 'order_id': 'nunique',
 'total_payment': 'sum',
 'Academic': 'max',
 'Auto': 'max',
 'Fashion_Accessories': 'max',
 'Health_Beauty': 'max',
 'Home_Furniture': 'max',
 'Kids_Baby': 'max',
 'Other': 'max',
 'Pets': 'max',
 'Sports_Leisure': 'max',
 'Technology': 'max'}

In [77]:
final_olist_customer_data = final_olist_order_dataset.groupby(['customer_unique_id']).agg(agg_rules).reset_index()

final_olist_customer_data

,customer_unique_id,days_last_purchased,order_id,total_payment,Academic,Auto,Fashion_Accessories,Health_Beauty,Home_Furniture,Kids_Baby,Other,Pets,Sports_Leisure,Technology
0,0000366f3b9a7992bf8c76cfdf3221e2,160,1,141.90,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,163,1,27.19,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0000f46a3911fa3c0805444483337064,585,1,86.22,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,369,1,43.62,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0004aac84e0df4da2b147fca70cf8255,336,1,196.89,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93352,fffcf5a5ff07b0908bd4e2dbc735a684,495,1,2067.42,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
93353,fffea47cd6d3cc0a88bd621562a9d061,310,1,84.58,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
93354,ffff371b4d645b6ecea244b27531430a,617,1,112.46,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
93355,ffff5962728ec6157033ef9805bacc48,168,1,133.69,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [84]:
# Frequency = 4 => 4 Items bought, but can be same category!!
final_olist_customer_data = final_olist_customer_data.rename(columns={"days_last_purchased":"Recency",
                                                                      "order_id":"Frequency",
                                                                      "total_payment":"Monetary"})
final_olist_customer_data.head()

,customer_unique_id,Recency,Frequency,Monetary,Academic,Auto,Fashion_Accessories,Health_Beauty,Home_Furniture,Kids_Baby,Other,Pets,Sports_Leisure,Technology
0,0000366f3b9a7992bf8c76cfdf3221e2,160,1,141.90,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,163,1,27.19,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0000f46a3911fa3c0805444483337064,585,1,86.22,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,369,1,43.62,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0004aac84e0df4da2b147fca70cf8255,336,1,196.89,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
final_olist_customer_data.isna().sum()
final_olist_customer_data.shape

# No null Values: Data Cleaned since only delivered orders in data
# Final Shape: 93357, 14

(93357, 14)